# 中证800 V67 纯模型训练导出器

这个 notebook 只负责训练和维护可上传到 JoinQuant 的模型 pkl：

- 读取固定训练面板
- 按可配置训练窗口训练 full 特征 LGB direct 模型
- 导出兼容现有 JoinQuant 回测脚本的 bundle pkl
- 保存模型 manifest / feature manifest / upload list

它不做组合回测、不做策略筛选、不做健康监测。组合和验证继续放在 V61/V65/V66。

In [ ]:
import os
import gc
import json
import pickle
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 240)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)
    def _gen():
        every = max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item
    return _gen()


def display_df(df, n=20):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))

## 1. 配置

日常只需要改 `TRAIN_WINDOW_SPECS`。

`LABEL_BOUNDARY_MODE` 有两个模式：

- `legacy_rebalance`：复刻 V61 口径，按 `rebalance_date <= train_end` 入样本，可能包含 `next_date` 超过 `train_end` 的标签。
- `label_end_safe`：更严格，要求 `next_date <= train_end`，更接近真实上线时“标签已经完全可见”的训练边界。

为了和前面 V61/V65/V66 结果对齐，默认先用 `legacy_rebalance`。

In [ ]:
# =========================
# Paths
# =========================
PROJECT_DIR = Path("/Users/youzou/Documents/New project/quant-research/机器学习策略")
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
MODEL_DIR = PROJECT_DIR / "models"
MANIFEST_DIR = PROJECT_DIR / "manifests"
RUN_OUT_DIR = PROJECT_DIR / "csi800_ml_v67_model_trainer_outputs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
RUN_OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("train_csi800_factor_v40_data_enhancement.csv"),
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement.csv",
    Path.home() / "Downloads" / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    Path.home() / "Downloads" / "train_csi800_factor_v40_data_enhancement.csv",
]
DATA_PATH_OVERRIDE = None

# =========================
# Training setup
# =========================
TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
INDUSTRY_COL = "industry_bucket"
BENCHMARK = "000906.XSHG"
MODEL_FAMILY = "v67_pure_model_trainer"
MODEL_VERSION_PREFIX = "v67"

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
TOP_N_CANDIDATES = 30
STOCK_NUM = 8
PORTFOLIO_RULE = "top8_board_cap"
BOARD_CAPS = {"chinext": 3, "star": 2}
BOARD_CAPS_TEXT = ";".join(["%s:%s" % (k, BOARD_CAPS[k]) for k in sorted(BOARD_CAPS)])
INDUSTRY_CAP_RATIO = 0.20

# Keep legacy_rebalance to reproduce V61/V66 model evidence. Use label_end_safe for stricter live-style exports.
LABEL_BOUNDARY_MODE = "legacy_rebalance"  # legacy_rebalance / label_end_safe

# Full-only by default. Keep this list small and intentional.
TRAIN_WINDOW_SPECS = [
    {"tag": "2019_2023", "train_start": "2019-01-01", "train_end": "2023-12-31", "note": "first OOS from 2024"},
    {"tag": "2019_2024", "train_start": "2019-01-01", "train_end": "2024-12-31", "note": "first OOS from 2025; current fixed-anchor candidate"},
    {"tag": "2019_2025", "train_start": "2019-01-01", "train_end": "2025-12-31", "note": "first OOS from 2026; latest expanding candidate"},
    # Examples you can enable later:
    # {"tag": "rolling60_2021_2025", "train_start": "2021-01-01", "train_end": "2025-12-31", "note": "rolling 60m latest"},
    # {"tag": "rolling72_2020_2025", "train_start": "2020-01-01", "train_end": "2025-12-31", "note": "rolling 72m latest"},
]

EXPORT_MODELS = True
SAVE_COPY_IN_RUN_OUT_DIR = True
PICKLE_PROTOCOL = 2

print("PROJECT_DIR:", PROJECT_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("RUN_OUT_DIR:", RUN_OUT_DIR)
print("LABEL_BOUNDARY_MODE:", LABEL_BOUNDARY_MODE)
print("TRAIN_WINDOW_SPECS:", len(TRAIN_WINDOW_SPECS))

## 2. Full 特征与 LGB 参数

这里固定为当前主线 full feature stack。后续如果要做特征实验，另开实验 notebook；这个 notebook 只维护生产候选模型。

In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio",
    "book_to_price_ratio",
    "earnings_yield",
    "sales_to_price_ratio",
    "cash_earnings_to_price_ratio",
    "earnings_to_price_ratio",
    "roe_ttm",
    "roa_ttm",
    "gross_profit_ttm",
    "operating_profit_to_total_profit",
    "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage",
    "adjusted_profit_to_total_profit",
    "ACCA",
    "growth",
    "net_working_capital",
    "operating_profit_per_share",
    "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share",
    "super_quick_ratio",
    "MLEV",
    "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio",
    "momentum",
    "Rank1M",
    "sharpe_ratio_60",
    "Variance20",
    "liquidity",
    "beta",
    "ATR6",
    "MFI14",
    "DAVOL10",
    "VOL10",
    "VMACD",
    "VOSC",
    "Skewness20",
    "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60",
    "liq_paused_count_20",
    "px_close_to_ma60",
    "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m",
    "ts_Rank1M_rank_chg_1m",
]

FULL_FEATURE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS

FEATURE_VARIANTS = [
    {
        "feature_variant": "full",
        "description": "V46/V56/V61 full hybrid-light feature stack",
        "candidate_cols": list(FULL_FEATURE_COLS),
    },
]

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}

feature_manifest_df = pd.DataFrame([
    {
        "feature_variant": v["feature_variant"],
        "candidate_feature_count": len(v["candidate_cols"]),
        "description": v["description"],
        "candidate_features": ",".join(v["candidate_cols"]),
    }
    for v in FEATURE_VARIANTS
])
feature_manifest_df.to_csv(RUN_OUT_DIR / "v67_feature_variant_manifest.csv", index=False)
display_df(feature_manifest_df)

## 3. Helper 函数

In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def resolve_data_path():
    if DATA_PATH_OVERRIDE:
        p = Path(DATA_PATH_OVERRIDE)
        if p.exists():
            return p
        raise IOError("DATA_PATH_OVERRIDE not found: %s" % p)
    candidates = [Path(x) for x in DATA_CANDIDATES]
    for p in candidates:
        if p.exists():
            return p
    searched = [str(p.resolve()) for p in candidates]
    raise IOError("training data csv not found. Put it in one of these paths: %s" % searched)


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.normalize()
    return out


def first_existing(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in progress_iter(range(len(feature_cols)), total=len(feature_cols), desc="corr graph", leave=False):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []

    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)

    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    if len(cols) == 0:
        raise ValueError("no candidate feature exists in train data")
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    y = d[target_col].astype(float).copy()
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index


def split_diag_valid(train_df):
    months = sorted(pd.to_datetime(train_df[DATE_COL].dropna().unique()))
    if len(months) <= 8:
        return train_df.copy(), train_df.copy()
    n_valid = max(6, int(round(len(months) * 0.20)))
    valid_months = set(months[-min(n_valid, len(months) - 1):])
    fit = train_df[~train_df[DATE_COL].isin(valid_months)].copy()
    valid = train_df[train_df[DATE_COL].isin(valid_months)].copy()
    if fit.empty or valid.empty:
        return train_df.copy(), train_df.copy()
    return fit, valid


def normalize_train_spec(spec):
    out = dict(spec)
    out["train_start"] = pd.Timestamp(out.get("train_start", "2019-01-01"))
    out["train_end"] = pd.Timestamp(out["train_end"])
    out["test_start"] = pd.Timestamp(out.get("test_start", out["train_end"] + pd.offsets.MonthBegin(1)))
    if not out.get("tag"):
        out["tag"] = "%s_%s" % (out["train_start"].strftime("%Y%m%d"), out["train_end"].strftime("%Y%m%d"))
    return out


def make_train_df(df_all, spec):
    spec = normalize_train_spec(spec)
    mask = (df_all[DATE_COL] >= spec["train_start"]) & (df_all[DATE_COL] <= spec["train_end"])
    if LABEL_BOUNDARY_MODE == "label_end_safe":
        mask = mask & (df_all["next_date"] <= spec["train_end"])
    elif LABEL_BOUNDARY_MODE != "legacy_rebalance":
        raise ValueError("unknown LABEL_BOUNDARY_MODE: " + str(LABEL_BOUNDARY_MODE))
    return df_all[mask].copy()


def needs_v4_adapter(feature_cols):
    adapter_cols = set(HYBRID_LIGHT_EXTRA_COLS)
    return any(c in adapter_cols for c in feature_cols)


def train_direct_lgb(train_df, feature_cols):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    X_train, y_train, fill_values, train_index = prepare_xy(train_df, feature_cols, TARGET_COL)
    if len(X_train) == 0:
        raise ValueError("empty training matrix")
    model = lgb.train(
        params,
        lgb.Dataset(X_train, label=y_train),
        num_boost_round=max(1, int(FIXED_ITER)),
    )
    pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    train_rank_ic = safe_rank_ic(y_train, pred)
    return {
        "model": model,
        "fill_values": fill_values,
        "train_rows": int(len(X_train)),
        "train_rank_ic": train_rank_ic,
    }


def make_model_id(spec, feature_variant):
    spec = normalize_train_spec(spec)
    return "%s_%s_%s_start%s_cutoff%s_fixed%s_%s" % (
        MODEL_VERSION_PREFIX,
        feature_variant,
        spec["tag"],
        spec["train_start"].strftime("%Y%m%d"),
        spec["train_end"].strftime("%Y%m%d"),
        int(FIXED_ITER),
        LABEL_BOUNDARY_MODE,
    )

## 4. 加载训练面板

In [ ]:
def load_dataset(path):
    df = pd.read_csv(path)
    df = safe_to_datetime(df, [DATE_COL, "feature_date", "next_date"])
    if STOCK_COL not in df.columns:
        for alt in ["code", "security", "order_book_id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: STOCK_COL})
                break
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = df["raw_return_1m"] - df["benchmark_csi800_1m"]
        else:
            raise ValueError("target column not found: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    need = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, "feature_date", "next_date"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[STOCK_COL, DATE_COL, TARGET_COL]).copy()
    return df


DATA_PATH = resolve_data_path()
df_all = load_dataset(DATA_PATH)

print("DATA_PATH:", DATA_PATH)
print("loaded:", df_all.shape)
print("date range:", df_all[DATE_COL].min(), "->", df_all[DATE_COL].max())
print("next_date range:", df_all["next_date"].min(), "->", df_all["next_date"].max())
print("target:", TARGET_COL)
display_df(df_all[[TARGET_COL]].describe().T)

availability_rows = []
for v in FEATURE_VARIANTS:
    configured = list(v["candidate_cols"])
    available = [c for c in configured if c in df_all.columns]
    missing = [c for c in configured if c not in df_all.columns]
    availability_rows.append({
        "feature_variant": v["feature_variant"],
        "configured": len(configured),
        "available": len(available),
        "missing": ",".join(missing),
    })
feature_availability_df = pd.DataFrame(availability_rows)
feature_availability_df.to_csv(RUN_OUT_DIR / "v67_feature_availability.csv", index=False)
display_df(feature_availability_df)

## 5. 训练并导出 pkl

导出的 pkl bundle 与现有 JoinQuant 回测脚本兼容：`base_model`、`base_feature_cols`、`base_fill_values`、`overlay_mode=direct`。

In [ ]:
def export_bundle(model_id, spec, variant, trained, feature_cols, removed_cols, diag_rank_ic, train_df):
    spec = normalize_train_spec(spec)
    model_file = "model_candidate_%s.pkl" % model_id
    model_path = MODEL_DIR / model_file
    run_copy_path = RUN_OUT_DIR / model_file
    bundle = {
        "objective": "v210_refit_fixed_iter_overlay",
        "research_version": model_id,
        "benchmark": BENCHMARK,
        "train_start": spec["train_start"],
        "train_end": spec["train_end"],
        "test_start": spec["test_start"],
        "label_end": spec["train_end"],
        "label_boundary_mode": LABEL_BOUNDARY_MODE,
        "require_label_end_within_train": bool(LABEL_BOUNDARY_MODE == "label_end_safe"),
        "legacy_unsealed_boundary": bool(LABEL_BOUNDARY_MODE == "legacy_rebalance"),
        "model_family": MODEL_FAMILY,
        "feature_variant": variant["feature_variant"],
        "target_col": TARGET_COL,
        "target_note": "V67 pure model trainer: direct LGB alpha_1m, fixed iteration, no early stopping",
        "data_file": str(DATA_PATH),
        "protocol": "v67_pure_model_training_export",
        "training_policy": "fixed_window_configurable",
        "param_set": "v46_base_ff10_original",
        "base_params": dict(BASE_PARAMS_FF10),
        "base_model": trained["model"],
        "base_feature_cols": list(feature_cols),
        "base_fill_values": dict(trained["fill_values"]),
        "base_best_iter": int(FIXED_ITER),
        "model_iter": int(FIXED_ITER),
        "fixed_iter": int(FIXED_ITER),
        "base_inner_metrics": {
            "train_rank_ic": float(trained["train_rank_ic"]) if not pd.isnull(trained["train_rank_ic"]) else np.nan,
            "diag_rank_ic": float(diag_rank_ic) if not pd.isnull(diag_rank_ic) else np.nan,
        },
        "base_removed_features": list(removed_cols),
        "residual_model": None,
        "residual_feature_cols": [],
        "residual_fill_values": {},
        "overlay_weight": 0.0,
        "overlay_mode": "direct",
        "top_n_candidates": TOP_N_CANDIDATES,
        "stock_num": STOCK_NUM,
        "portfolio_rule": PORTFOLIO_RULE,
        "board_caps": dict(BOARD_CAPS),
        "board_caps_text": BOARD_CAPS_TEXT,
        "industry_cap_ratio": INDUSTRY_CAP_RATIO,
        "requires_v4_feature_adapter": bool(needs_v4_adapter(feature_cols)),
        "requires_industry_relative_adapter": False,
        "uses_time_weight": False,
        "uses_sample_weight": False,
        "uses_current_valid_for_training": False,
        "final_role": "v67_pure_training_candidate",
        "train_row_count": int(len(train_df)),
        "train_month_count": int(train_df[DATE_COL].nunique()),
        "max_train_rebalance_date": str(train_df[DATE_COL].max().date()),
        "max_train_next_date": str(train_df["next_date"].max().date()) if "next_date" in train_df.columns else "",
        "note": spec.get("note", ""),
    }
    if EXPORT_MODELS:
        with open(model_path, "wb") as f:
            pickle.dump(bundle, f, protocol=PICKLE_PROTOCOL)
        if SAVE_COPY_IN_RUN_OUT_DIR:
            shutil.copy2(model_path, run_copy_path)
    return model_path, run_copy_path if SAVE_COPY_IN_RUN_OUT_DIR else None, model_file


def train_one_spec_variant(df_all, spec, variant):
    spec = normalize_train_spec(spec)
    train_df = make_train_df(df_all, spec)
    if train_df.empty:
        raise ValueError("empty train_df for %s" % spec["tag"])
    diag_fit_df, diag_valid_df = split_diag_valid(train_df)
    feature_cols, removed_cols = select_features_train_only(diag_fit_df, variant["candidate_cols"])
    trained = train_direct_lgb(train_df, feature_cols)
    X_valid, y_valid, _, _ = prepare_xy(diag_valid_df, feature_cols, TARGET_COL, trained["fill_values"])
    valid_pred = np.asarray(trained["model"].predict(X_valid[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    diag_rank_ic = safe_rank_ic(y_valid, valid_pred)
    model_id = make_model_id(spec, variant["feature_variant"])
    model_path, run_copy_path, model_file = export_bundle(
        model_id, spec, variant, trained, feature_cols, removed_cols, diag_rank_ic, train_df
    )
    row = {
        "model_id": model_id,
        "model_family": MODEL_FAMILY,
        "feature_variant": variant["feature_variant"],
        "tag": spec["tag"],
        "note": spec.get("note", ""),
        "train_start": spec["train_start"],
        "train_end": spec["train_end"],
        "test_start": spec["test_start"],
        "label_boundary_mode": LABEL_BOUNDARY_MODE,
        "train_rows": int(len(train_df)),
        "train_months": int(train_df[DATE_COL].nunique()),
        "max_train_rebalance_date": str(train_df[DATE_COL].max().date()),
        "max_train_next_date": str(train_df["next_date"].max().date()) if "next_date" in train_df.columns else "",
        "candidate_feature_count": len(variant["candidate_cols"]),
        "feature_count": len(feature_cols),
        "removed_feature_count": len(removed_cols),
        "train_rank_ic": trained["train_rank_ic"],
        "diag_rank_ic": diag_rank_ic,
        "model_file": model_file,
        "model_path": str(model_path),
        "run_copy_path": str(run_copy_path) if run_copy_path is not None else "",
        "feature_cols": ",".join(feature_cols),
        "removed_features": ",".join(removed_cols),
        "top_n_candidates": TOP_N_CANDIDATES,
        "stock_num": STOCK_NUM,
        "portfolio_rule": PORTFOLIO_RULE,
        "board_caps": BOARD_CAPS_TEXT,
    }
    del trained
    gc.collect()
    return row


manifest_rows = []
jobs = [(spec, variant) for spec in TRAIN_WINDOW_SPECS for variant in FEATURE_VARIANTS]
for spec, variant in progress_iter(jobs, total=len(jobs), desc="train/export models"):
    row = train_one_spec_variant(df_all, spec, variant)
    manifest_rows.append(row)
    print("exported", row["model_file"], "rows", row["train_rows"], "features", row["feature_count"], "diag_ic", row["diag_rank_ic"])

model_manifest_df = pd.DataFrame(manifest_rows)
model_manifest_df.to_csv(RUN_OUT_DIR / "v67_model_manifest.csv", index=False)
model_manifest_df.to_csv(MANIFEST_DIR / "v67_model_manifest.csv", index=False)
display_df(model_manifest_df)

## 6. 生成聚宽上传清单

把这些 `model_file` 上传到聚宽 `test/` 目录后，就可以在 V66 回测脚本里配置候选模型。

In [ ]:
upload_rows = []
for _, r in model_manifest_df.iterrows():
    upload_rows.append({
        "model_name": r["model_id"],
        "model_file": r["model_file"],
        "local_model_path": r["model_path"],
        "joinquant_file": "test/" + r["model_file"],
        "train_start": r["train_start"],
        "train_end": r["train_end"],
        "label_boundary_mode": r["label_boundary_mode"],
        "feature_count": r["feature_count"],
        "train_rank_ic": r["train_rank_ic"],
        "diag_rank_ic": r["diag_rank_ic"],
        "portfolio_rule": r["portfolio_rule"],
        "stock_num": r["stock_num"],
        "board_caps": r["board_caps"],
    })

upload_list_df = pd.DataFrame(upload_rows)
upload_list_df.to_csv(RUN_OUT_DIR / "v67_joinquant_upload_list.csv", index=False)
upload_list_df.to_csv(MANIFEST_DIR / "v67_joinquant_upload_list.csv", index=False)

print("saved:")
for path in [
    RUN_OUT_DIR / "v67_model_manifest.csv",
    RUN_OUT_DIR / "v67_joinquant_upload_list.csv",
    MANIFEST_DIR / "v67_model_manifest.csv",
    MANIFEST_DIR / "v67_joinquant_upload_list.csv",
]:
    print("-", path)

display_df(upload_list_df)

## 7. 轻量完整性检查

这里只验证 pkl 是否能读回、bundle 关键字段是否存在。注意：本地环境如果缺 LightGBM 动态库，读回 pkl 可能失败；聚宽环境通常可以正常加载。

In [ ]:
RELOAD_CHECK = True

required_bundle_keys = [
    "objective",
    "base_model",
    "base_feature_cols",
    "base_fill_values",
    "overlay_mode",
    "top_n_candidates",
    "stock_num",
    "portfolio_rule",
]

check_rows = []
if RELOAD_CHECK:
    for _, r in progress_iter(model_manifest_df.iterrows(), total=len(model_manifest_df), desc="reload check"):
        path = Path(r["model_path"])
        row = {"model_file": r["model_file"], "path": str(path), "exists": path.exists(), "ok": False, "error": ""}
        try:
            with open(path, "rb") as f:
                bundle = pickle.load(f)
            missing = [k for k in required_bundle_keys if k not in bundle]
            row["missing_keys"] = ",".join(missing)
            row["ok"] = len(missing) == 0
            row["feature_count"] = len(bundle.get("base_feature_cols", []))
            row["overlay_mode"] = bundle.get("overlay_mode")
            row["portfolio_rule"] = bundle.get("portfolio_rule")
        except Exception as err:
            row["error"] = str(err)
        check_rows.append(row)

reload_check_df = pd.DataFrame(check_rows)
reload_check_df.to_csv(RUN_OUT_DIR / "v67_reload_check.csv", index=False)
display_df(reload_check_df)